# IGNIS — Next-Day Wildfire Spread Dataset Generation (v4)

**v4 arşivi.** v3'ün üzerine, projenin en büyük teşhis edilmiş sorununu hedef alan
dört bant ekler. `data/spread_v4/` içine indirilir — **v2 veya v3 ile aynı klasöre
KOYMAYIN**, bant sırası sözleşmedir ve karışık bir klasör sessizce bozuk eğitim verir.

---

## v4 neden var

v1 teşhisindeki **3. kök neden** şuydu: yamaların **%58,9'unda** *t*+1 gününde hiç
yangın pikseli yok, oysa aynı yamada *t* gününde ortalama **12,3** piksel yanıyor.
Yangın bir günde sönmedi — uydu o an göremedi.

Şimdiye kadar bunu bir veri kusuru sandık. Aslında **kendi kodumuzun ürettiği bir
kusurdu.** MODIS `FireMask` bandı gözlem kalitesini zaten kodluyor:

| Değer | Anlam | Gözlendi mi? |
|---|---|---|
| 0, 1, 2 | işlenmedi (girdi verisi yok) | **hayır** |
| 3 | su | evet |
| 4 | **bulut** | **hayır** |
| 5 | yangınsız kara | evet |
| 6 | bilinmiyor | **hayır** |
| 7, 8, 9 | yangın (düşük / orta / yüksek güven) | evet |

v2 ve v3'te `daily_fire_mask` şunu yapıyordu:

```python
fm.gte(7).unmask(0)      # bulut  -> 0 -> "yangın yok"
```

Yani **bulutun arkasındaki bir piksel, "yangın yok" olarak etiketleniyordu.** Ağı
bununla eğitmek, ona yangın davranışını değil bulut örüntüsünü öğretmek demek.

v4 bu bilgiyi atmıyor. `valid_next` ve `valid_next2` bantları, hedef maskesinin
gerçekten gözlenip gözlenmediğini söylüyor; eğitimde gözlenmemiş pikseller kayba
hiç girmiyor.

---

## v4'te yeni olan

| Bant | Ne | Neden |
|---|---|---|
| `valid_next` | *t*+1 hedefi gerçekten gözlendi mi | **3. kök nedenin doğrudan çözümü.** Bulutlu/eksik pikseller artık sahte negatif değil, maskeli. |
| `valid_next2` | *t*+2 hedefi gözlendi mi | Aynısı, ±1 gün hedefinin ikinci yarısı için. |
| `burn_age` | Bu piksel en son kaç gün önce yandı (0–14, hiç yanmadıysa 14) | Yangın, yakıtı tükenmiş araziye yayılmaz. Ağın bunu görmesi gerek. |
| `days_since_rain` | Son ölçülebilir yağıştan (>1 mm) bu yana geçen gün (0–30) | İnce yakıt nemi. 24 saatlik yağış piksellerin %91,5'inde sıfır; "ne zamandır kuru" bilgisi bunu taşımıyor. |

Ayrıca `valid` bandı artık bugünün yangın gözlemini de içeriyor — yalnızca çevresel
bantların maskesini değil.

---

## Bant sözleşmesi (v4 — 26 bant)

```
ndvi  lst  air_temp  humidity  vpd
wind_speed  wind_u  wind_v
precip  precip_7d  precip_30d  days_since_rain
soil_moisture
elevation  slope  aspect  landcover
burn_age
fire_prev2  fire_prev1  fire              <- 21 girdi bandı

fire_next  fire_next2                      <- hedef
valid  valid_next  valid_next2             <- gözlem geçerliliği
```

Bu sıra **sözleşmedir** ve şu üç yerde birebir aynı olmalıdır:
bu not defteri, `src/gee_config.py`, `src/config.py`.

---

## Çalıştırma

1. Bölüm 1–6'yı sırayla çalıştırın (GEE kimlik doğrulaması ilk hücrede).
2. Bölüm 7 ve 8 dışa aktarım görevlerini açar. `SUBMIT_LIMIT = 400`, yani her
   çalıştırmada en fazla 400 gün gönderilir — **bitene kadar 7 ve 8'i tekrar
   çalıştırın.** Not defteri yeniden başlatılabilirdir; tamamlanmış günleri atlar.
3. Bölüm 9'daki talimatla Drive'daki `GEE_FireSpread_v4` klasörünü indirin ve
   `data/spread_v4/` içine koyun.

Görev açıklamaları `firespread_v4_YYYYMMDD` biçimindedir; devam taraması sürüme
göre ayrışır, böylece v2/v3 görevleri v4 sanılmaz. İndirilen dosya adları
`firespread_YYYYMMDD.tfrecord.gz` olarak kalır.


## 1 — Environment / Ortam

In [ ]:
# Colab: run once per session. / Colab: oturum başına bir kez çalıştırın.
!pip install -q earthengine-api geemap

In [ ]:
import datetime
import ee
import geemap

# Replace with your own Earth Engine project ID.
# Kendi Earth Engine proje kimliğinizi yazın.
EE_PROJECT = 'ignisai-496207'

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print(f'Earth Engine ready / hazir  —  project: {EE_PROJECT}')

## 2 — Configuration / Konfigürasyon

Every constant that defines the dataset lives in this cell. The band order declared in
`INPUT_BANDS` is **contractual**: it must match `src/config.py → SPREAD_INPUT_BANDS` and
`src/gee_config.py → GEEConfig.INPUT_BANDS` exactly, because the training pipeline
reconstructs the channel axis from this order alone.

Veri setini tanımlayan tüm sabitler bu hücrededir. `INPUT_BANDS` içindeki bant sırası
**sözleşmeseldir**: `src/config.py → SPREAD_INPUT_BANDS` ve
`src/gee_config.py → GEEConfig.INPUT_BANDS` ile birebir aynı olmalıdır, çünkü eğitim
hattı kanal eksenini yalnızca bu sıraya bakarak yeniden kurar.

In [ ]:
# ─────────────────────── STUDY AREA / ÇALIŞMA ALANI ───────────────────────
REGION = (ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
            .filter(ee.Filter.eq('country_na', 'Turkey')))

# Analysis grid: UTM zone 35N at 1 km. Metric and square over Türkiye.
# Analiz gridi: UTM 35N, 1 km. Türkiye üzerinde metrik ve kare.
SCALE = 1000                                   # metres per pixel / metre / piksel
PROJ  = ee.Projection('EPSG:32635').atScale(SCALE)

# ─────────────────────── PATCH GEOMETRY / YAMA GEOMETRİSİ ─────────────────
PATCH_RADIUS = 32                              # -> 65 x 65
PATCH_SIZE   = 2 * PATCH_RADIUS + 1

# ─────────────────────── TEMPORAL COVERAGE / ZAMANSAL KAPSAM ──────────────
FIRE_SEASON = (6, 10)                          # June - October / Haziran - Ekim
YEARS       = [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

# ─────────────────────── ACTIVE FIRE / AKTİF YANGIN ───────────────────────
# MODIS FireMask classes / MODIS FireMask sınıfları:
#   0,1,2 not processed (no input data)  islenmedi          -> NOT observed
#   3     water                          su                 -> observed
#   4     cloud                          bulut              -> NOT observed
#   5     non-fire land                  yanginsiz kara     -> observed
#   6     unknown                        bilinmiyor         -> NOT observed
#   7,8,9 fire: low / nominal / high     yangin             -> observed
FIRE_CONFIDENCE    = 7
OBSERVED_CLASSES   = (3, 5, 7, 8, 9)   # a real look at the ground / gercek gozlem
MIN_FIRE_PIXELS    = 5      # minimum active fire pixels for a day to be exported
MAX_POINTS_PER_DAY = 150    # patches sampled per fire day / gün başına yama sayısı
SEED               = 42

# ─────────────────────── LOOKBACK WINDOWS / GERİYE BAKIŞ ──────────────────
BURN_AGE_MAX  = 14   # days; a pixel not burned within this window reports the cap
RAIN_AGE_MAX  = 30   # days since last measurable rain, capped
RAIN_MM       = 1.0  # "measurable" threshold / "olculebilir" esigi

# ─────────────────────── BAND CONTRACT / BANT SÖZLEŞMESİ ──────────────────
INPUT_BANDS = [
    'ndvi',            #  1 vegetation index          bitki örtüsü indeksi
    'lst',             #  2 land surface temp. [degC] arazi yüzey sıcaklığı
    'air_temp',        #  3 2 m air temp.     [degC]  hava sıcaklığı
    'humidity',        #  4 relative humidity [%]     bağıl nem
    'vpd',             #  5 vapour pressure deficit [kPa]  buhar basıncı açığı
    'wind_speed',      #  6 wind magnitude    [m/s]   rüzgâr hızı
    'wind_u',          #  7 eastward wind     [m/s]   rüzgâr doğu bileşeni
    'wind_v',          #  8 northward wind    [m/s]   rüzgâr kuzey bileşeni
    'precip',          #  9 precipitation, 24 h [mm]  yağış, 24 saat
    'precip_7d',       # 10 precipitation, 7 d  [mm]  birikimli yağış, 7 gün
    'precip_30d',      # 11 precipitation, 30 d [mm]  birikimli yağış, 30 gün
    'days_since_rain', # 12 days since >1 mm   [d]    son yağıştan bu yana gün   (v4)
    'soil_moisture',   # 13 soil water        [m3/m3] toprak nemi
    'elevation',       # 14 elevation         [m]     yükseklik
    'slope',           # 15 slope             [deg]   eğim
    'aspect',          # 16 aspect            [deg]   bakı
    'landcover',       # 17 IGBP fuel class           yakıt sınıfı
    'burn_age',        # 18 days since last burn [d]  son yanıştan bu yana gün   (v4)
    'fire_prev2',      # 19 fire mask, day t-2        iki gün önceki yangın maskesi
    'fire_prev1',      # 20 fire mask, day t-1        dünkü yangın maskesi
    'fire',            # 21 fire mask, day t          bugünün yangın maskesi
]
TARGET_BANDS = [
    'fire_next',       # fire mask, day t+1           yarının yangın maskesi
    'fire_next2',      # fire mask, day t+2           öbür günün yangın maskesi
    'valid',           # inputs AND today's fire observed  girdiler ve bugün gözlendi
    'valid_next',      # t+1 target genuinely observed     t+1 gerçekten gözlendi (v4)
    'valid_next2',     # t+2 target genuinely observed     t+2 gerçekten gözlendi (v4)
]
ALL_BANDS = INPUT_BANDS + TARGET_BANDS
META_COLS = ['lon', 'lat', 'date']

# ─────────────────────── GROWTH CLASSES / BÜYÜME SINIFLARI ────────────────
# r = N(t+1) / max(N(t), 1)
GROW_RATIO = 1.25          # r > 1.25              -> growing    / büyüyor
STABLE_LOW = 0.75          # 0.75 <= r <= 1.25     -> stable     / sabit
#                            r < 0.75              -> extinguishing / sönüyor

# ─────────────────────── EXPORT / DIŞA AKTARIM ────────────────────────────
# A NEW folder, as always. v2 (17 bands), v3 (22) and v4 (26) must never share a
# directory: the loader rebuilds the channel axis from band ORDER alone, so a
# record from the wrong schema is silently misread rather than rejected.
# HER ZAMAN YENİ bir klasör. v2 (17 bant), v3 (22) ve v4 (26) asla aynı dizini
# paylaşmamalıdır; yükleyici kanal eksenini yalnızca bant SIRASINDAN kurar.
DRIVE_FOLDER = 'GEE_FireSpread_v4'
# Version tag. Earth Engine task descriptions are namespaced with this so the
# resume scan cannot mistake a v2/v3 task for a v4 one: without it every schema
# produces the identical description 'firespread_YYYYMMDD', and a previous run's
# SUCCEEDED tasks mark the entire date range as already done.
# Sürüm etiketi. EE görev açıklamaları bununla ayrılır; aksi hâlde her şema aynı
# 'firespread_YYYYMMDD' açıklamasını üretir ve önceki çalıştırmanın SUCCEEDED
# görevleri tüm tarih aralığını "bitti" gibi gösterir.
VERSION_TAG  = 'v4'
SUBMIT_LIMIT = 400   # export tasks submitted per notebook run; re-run to continue
                     # her çalıştırmada açılacak görev sayısı; devam için tekrar çalıştırın

print(f'Patch      : {PATCH_SIZE} x {PATCH_SIZE} px @ {SCALE} m')
print(f'Bands      : {len(INPUT_BANDS)} input + {len(TARGET_BANDS)} target = {len(ALL_BANDS)}')
print(f'Years      : {YEARS[0]}-{YEARS[-1]}, months {FIRE_SEASON[0]}-{FIRE_SEASON[1]}')
print(f'Drive      : {DRIVE_FOLDER}   (tasks: firespread_{VERSION_TAG}_YYYYMMDD)')
assert len(ALL_BANDS) == 26, 'band contract drift / bant sözleşmesi kaymış'


## 3 — Data availability / Veri erişilebilirliği

MODIS thermal anomaly products reach Earth Engine with a latency of a few days, and the
target definition requires day *t*+2. The last exportable sample date is therefore
`last(MOD14A1) − 2 days`. This cell queries the collections directly rather than assuming
a fixed end date, so the notebook remains correct as new data is published.

MODIS termal anomali ürünleri Earth Engine'e birkaç gün gecikmeyle ulaşır ve hedef tanımı
*t*+2 gününü gerektirir. Bu nedenle dışa aktarılabilir son örnek tarihi
`son(MOD14A1) − 2 gün`'dür. Bu hücre sabit bir bitiş tarihi varsaymak yerine koleksiyonları
doğrudan sorgular; böylece yeni veri yayımlandıkça notebook doğru kalmaya devam eder.

In [ ]:
def collection_span(cid):
    '''Return (first, last) acquisition dates of a collection as ISO strings.'''
    c = ee.ImageCollection(cid)
    lo = ee.Date(c.aggregate_min('system:time_start')).format('YYYY-MM-dd')
    hi = ee.Date(c.aggregate_max('system:time_start')).format('YYYY-MM-dd')
    return ee.List([lo, hi]).getInfo()


CHECK = [
    ('MODIS/061/MOD14A1',            'active fire (Terra)'),
    ('MODIS/061/MYD14A1',            'active fire (Aqua)'),
    ('MODIS/061/MOD13Q1',            'NDVI'),
    ('MODIS/061/MOD11A1',            'land surface temperature'),
    ('ECMWF/ERA5_LAND/DAILY_AGGR',   'meteorology'),
    ('UCSB-CHG/CHIRPS/DAILY',        'precipitation'),
    ('MODIS/061/MCD12Q1',            'land cover'),
]

print(f'{"collection":<32}{"purpose":<28}{"first":<12}{"last":<12}')
print('-' * 84)
spans = {}
for cid, purpose in CHECK:
    lo, hi = collection_span(cid)
    spans[cid] = (lo, hi)
    print(f'{cid:<32}{purpose:<28}{lo:<12}{hi:<12}')

# Latest sample date we can build a t+2 target for.
# t+2 hedefi kurabileceğimiz en son örnek tarihi.
_last_fire = datetime.date.fromisoformat(spans['MODIS/061/MOD14A1'][1])
LAST_SAMPLE_DATE = _last_fire - datetime.timedelta(days=2)
print(f'\nLast usable sample date / son kullanilabilir ornek tarihi: {LAST_SAMPLE_DATE}')

## 4 — Raster construction / Raster kurulumu

### Fire mask / Yangın maskesi

Terra and Aqua cross the equator at different local times, so merging both `FireMask`
bands with a pixel-wise maximum roughly doubles the daily detection opportunity. Pixels
whose confidence class is below `FIRE_CONFIDENCE` — including the water, cloud and
not-processed classes — are treated as non-fire.

Terra ve Aqua ekvatoru farklı yerel saatlerde geçer; bu yüzden iki `FireMask` bandını
piksel bazında maksimumla birleştirmek günlük tespit şansını kabaca ikiye katlar. Güven
sınıfı `FIRE_CONFIDENCE` altındaki pikseller — su, bulut ve işlenmemiş sınıfları dâhil —
yangın dışı sayılır.

### Temporal compositing / Zamansal derleme

`MOD13Q1` has a 16-day revisit, so the most recent composite within a 32-day lookback is
used. `MOD11A1` is daily but frequently cloud-obscured, so a 3-day mean is taken. This
follows Section 2.3 of the manuscript and is unchanged in this revision.

`MOD13Q1` 16 günlük tekrar ziyaret süresine sahiptir; bu yüzden 32 günlük geriye bakış
içindeki en yeni kompozit alınır. `MOD11A1` günlüktür ama sık sık bulut altında kalır,
bu yüzden 3 günlük ortalama alınır. Bu, makalenin 2.3 bölümüne uygundur ve bu sürümde
değişmemiştir.

In [ ]:
def fire_and_obs(date):
    """(fire mask, observation mask) for one day, Terra and Aqua merged.

    Tek gun icin (yangin maskesi, gozlem maskesi); Terra + Aqua birlesik.

    THIS IS THE v4 CHANGE. v2/v3 did `fm.gte(7).unmask(0)`, which collapses three
    different situations into the single value 0:
        - the ground was observed and was not burning   (FireMask 3 or 5)
        - the ground was hidden by cloud                (FireMask 4)
        - the granule was never processed               (FireMask 0, 1, 2)
    Only the first is "no fire". Treating the other two as "no fire" is how 58.9 %
    of v1 patches ended up with an empty t+1 target while 12.3 pixels burned on t.
    The observation mask keeps the three apart.

    v4 DEGISIKLIGI BUDUR. v2/v3 `fm.gte(7).unmask(0)` yapiyordu ve bu, uc farkli
    durumu tek bir 0 degerine indiriyordu: gozlendi-yanmiyordu, bulut vardi,
    hic islenmedi. Yalnizca birincisi "yangin yok" demektir.

    Terra and Aqua are merged with max(). Because 9 > 8 > 7 > 5 > 4 > 3 > 2 > 1 > 0,
    max() prefers a detection over a non-detection AND prefers a clear look over a
    clouded one -- exactly the desired precedence for both outputs.
    Terra ve Aqua max() ile birlestirilir; siralama geregi max(), tespiti
    tespitsizlige ve acik gorusu bulutluya tercih eder.
    """
    d0 = ee.Date(date)
    d1 = d0.advance(1, 'day')
    terra = ee.ImageCollection('MODIS/061/MOD14A1').filterDate(d0, d1).select('FireMask')
    aqua  = ee.ImageCollection('MODIS/061/MYD14A1').filterDate(d0, d1).select('FireMask')
    col   = terra.merge(aqua)

    # Absent granule -> constant 0 == "not processed", which the observation mask
    # correctly reports as unobserved.
    # Granul yoksa sabit 0 == "islenmedi"; gozlem maskesi bunu dogru bildirir.
    fm = ee.Image(ee.Algorithms.If(col.size().gt(0),
                                   col.max(),
                                   ee.Image.constant(0).rename('FireMask')))

    fire = fm.gte(FIRE_CONFIDENCE).rename('fire').toFloat()
    # observed = FireMask in {3, 5, 7, 8, 9}
    obs = (fm.eq(3).Or(fm.eq(5)).Or(fm.gte(FIRE_CONFIDENCE))).rename('obs').toFloat()
    return fire.clip(REGION), obs.clip(REGION)


def daily_fire_mask(date):
    """Binary active-fire mask only. / Yalnizca ikili aktif yangin maskesi.

    Kept so the visual-inspection section and any external caller still work.
    Gorsel inceleme bolumu ve disaridan cagrilar calissin diye tutuldu.
    """
    return fire_and_obs(date)[0].unmask(0)


def _days_since(col, flag_fn, d0, cap, name):
    """Days since the most recent image in `col` where flag_fn(img) is true.

    `col` icinde flag_fn(img) dogru olan en son goruntuden bu yana gecen gun.

    Implemented by stamping each qualifying pixel with its acquisition time and
    taking the per-pixel max, which needs ONE collection pass instead of one
    filter per day in the lookback window.
    Her uygun pikseli cekim zamaniyla damgalayip piksel bazinda max almak,
    her gun icin ayri filtre yerine TEK gecis gerektirir.
    """
    stamped = col.map(lambda img: flag_fn(img)
                      .multiply(ee.Number(ee.Image(img).date().millis()))
                      .rename('t'))
    last = ee.ImageCollection(stamped).max().unmask(0)
    age = d0.millis().subtract(last).divide(86400000)
    # last == 0 means "never in the window" -> report the cap.
    # last == 0 ise pencerede hic yok demektir -> tavan deger.
    return (last.gt(0).multiply(age)
            .add(last.eq(0).multiply(cap))
            .clamp(0, cap).rename(name).toFloat())


def burn_age(date):
    """Days since this pixel last burned, capped at BURN_AGE_MAX.
    Bu pikselin en son yandigi gunden bu yana gecen sure.

    Fire does not spread into ground whose fuel it has already consumed. A pixel
    that burned two days ago is close to unburnable; one that has not burned in
    two weeks is ordinary fuel. Without this the network has no way to tell the
    burnt interior of a fire from the unburnt land ahead of its front.
    Yangin, yakitini zaten tukettigi araziye yayilmaz. Bu bant olmadan ag, bir
    yanginin yanmis ic bolgesiyle cephesinin onundeki yanmamis araziyi ayiramaz.
    """
    d0 = ee.Date(date)
    col = (ee.ImageCollection('MODIS/061/MOD14A1')
             .merge(ee.ImageCollection('MODIS/061/MYD14A1'))
             .filterDate(d0.advance(-BURN_AGE_MAX, 'day'), d0)
             .select('FireMask'))
    return _days_since(col, lambda i: ee.Image(i).gte(FIRE_CONFIDENCE),
                       d0, BURN_AGE_MAX, 'burn_age').clip(REGION)


def days_since_rain(date):
    """Days since the last measurable rainfall, capped at RAIN_AGE_MAX.
    Son olculebilir yagistan bu yana gecen gun.

    The 24 h precipitation total is exactly zero on 91.4 % of pixels in the v2
    archive, so on its own it carries almost no information. "How long has it been
    dry" is the quantity that actually tracks fine-fuel moisture.
    24 saatlik yagis toplami v2 arsivinde piksellerin %91.4'unde tam olarak
    sifirdir ve tek basina neredeyse hic bilgi tasimaz.
    """
    d0 = ee.Date(date)
    col = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
             .filterDate(d0.advance(-RAIN_AGE_MAX, 'day'), d0)
             .select('precipitation'))
    return _days_since(col, lambda i: ee.Image(i).gt(RAIN_MM),
                       d0, RAIN_AGE_MAX, 'days_since_rain').clip(REGION)


def feature_stack(date):
    """The 18 environmental driver bands for one day (fire masks excluded).

    Bir gune ait 18 cevresel surucu bandi (yangin maskeleri haric).

    Bands remain MASKED where no valid observation exists; the mask is consumed by
    build_sample_image() to derive `valid` and only then replaced with zero.
    Gecerli gozlem olmayan yerlerde bantlar MASKELI kalir; bu maske
    build_sample_image() tarafindan `valid` icin kullanilir, sonra sifirlanir.
    """
    d0 = ee.Date(date)
    d1 = d0.advance(1, 'day')

    # --- Vegetation / Bitki ortusu -------------------------------------------------
    ndvi = (ee.ImageCollection('MODIS/061/MOD13Q1')
              .filterDate(d0.advance(-32, 'day'), d1).select('NDVI')
              .sort('system:time_start', False).first()
              .multiply(0.0001).rename('ndvi'))

    # --- Land surface temperature / Arazi yuzey sicakligi ---------------------------
    lst = (ee.ImageCollection('MODIS/061/MOD11A1')
             .filterDate(d0.advance(-3, 'day'), d1).select('LST_Day_1km')
             .mean().multiply(0.02).subtract(273.15).rename('lst'))

    # --- Meteorology / Meteoroloji --------------------------------------------------
    era = ee.Image(ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
                     .filterDate(d0.advance(-7, 'day'), d1)
                     .sort('system:time_start', False).first())

    air = era.select('temperature_2m').subtract(273.15).rename('air_temp')
    dew = era.select('dewpoint_temperature_2m').subtract(273.15)

    # Relative humidity from the Magnus relation, clipped to a physical range.
    a = dew.multiply(17.625).divide(dew.add(243.04))
    b = air.multiply(17.625).divide(air.add(243.04))
    humidity = a.subtract(b).exp().multiply(100).clamp(0, 100).rename('humidity')

    # Vapour pressure deficit: the drying power of the air, in kPa.
    # Buhar basinci acigi: havanin kurutma gucu, kPa.
    def _es(t):
        return t.multiply(17.27).divide(t.add(237.3)).exp().multiply(0.6108)
    vpd = _es(air).subtract(_es(dew)).max(0).rename('vpd')

    u     = era.select('u_component_of_wind_10m').rename('wind_u')
    v     = era.select('v_component_of_wind_10m').rename('wind_v')
    speed = u.hypot(v).rename('wind_speed')
    soil  = era.select('volumetric_soil_water_layer_1').rename('soil_moisture')

    # --- Precipitation / Yagis ------------------------------------------------------
    def _precip_sum(days, name):
        """Accumulated precipitation, CHIRPS with ERA5 fallback."""
        c = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
               .filterDate(d0.advance(-days, 'day'), d1).select('precipitation'))
        e = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
               .filterDate(d0.advance(-days, 'day'), d1)
               .select('total_precipitation_sum'))
        return ee.Image(ee.Algorithms.If(
            c.size().gt(0), c.sum(),
            ee.Image(ee.Algorithms.If(e.size().gt(0), e.sum().multiply(1000),
                                      ee.Image.constant(0))))).rename(name)

    precip     = _precip_sum(1,  'precip')
    precip_7d  = _precip_sum(7,  'precip_7d')
    precip_30d = _precip_sum(30, 'precip_30d')
    dry        = days_since_rain(d0)                       # v4

    # --- Terrain / Arazi ------------------------------------------------------------
    terr   = ee.Terrain.products(ee.Image('USGS/SRTMGL1_003'))
    elev   = terr.select('elevation').rename('elevation')
    slope  = terr.select('slope').rename('slope')
    aspect = terr.select('aspect').rename('aspect')

    # --- Fuel type / Yakit tipi -----------------------------------------------------
    yr     = d0.get('year')
    lc_col = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1')
    lc_yr  = lc_col.filter(ee.Filter.calendarRange(yr, yr, 'year'))
    lc = ee.Image(ee.Algorithms.If(lc_yr.size().gt(0),
                                   lc_yr.first(),
                                   lc_col.sort('system:time_start', False).first())
                  ).rename('landcover')

    # --- Fuel consumption history / Yakit tuketim gecmisi ---------------------------
    age = burn_age(d0)                                     # v4

    return (ndvi.addBands([lst, air, humidity, vpd, speed, u, v,
                           precip, precip_7d, precip_30d, dry, soil,
                           elev, slope, aspect, lc, age])
                .toFloat().clip(REGION))


### The `valid` band / `valid` bandı

`valid` is the pixel-wise minimum of the 13 environmental band masks, evaluated **before**
`unmask(0)`. It is 1 only where every input variable was genuinely observed, and 0 where a
value was fabricated — outside the national border, or where cloud removed a retrieval.

Measurement of the previous archive showed an identical 15 % zero rate across all
environmental bands, confirming that these fabricated zeros were being presented to the
network as if they were physical measurements. A relative humidity of 0 % and an
unobserved pixel are numerically indistinguishable without this band.

`valid`, 13 çevresel bant maskesinin piksel bazında minimumudur ve `unmask(0)`
uygulanmadan **önce** hesaplanır. Yalnızca her girdi değişkeninin gerçekten gözlendiği
yerde 1, bir değerin uydurulduğu yerde 0'dır — ülke sınırı dışında veya bulutun ölçümü
engellediği yerlerde.

Önceki arşivin ölçümü, tüm çevresel bantlarda birebir aynı %15 sıfır oranı gösterdi; bu
da uydurulmuş sıfırların ağa fiziksel ölçümmüş gibi sunulduğunu doğruladı. Bu bant
olmadan %0 bağıl nem ile gözlenmemiş bir piksel sayısal olarak ayırt edilemez.

In [ ]:
def build_sample_image(date):
    """26-band export stack for one day, reprojected onto the common grid.

    Bir gune ait 26 bantli disa aktarim yigini, ortak gride yeniden projekte edilmis.

    Band order is fixed by ALL_BANDS and is contractual with the training pipeline.
    Bant sirasi ALL_BANDS ile sabitlenir ve egitim hattiyla sozlesmeseldir.
    """
    d0 = ee.Date(date)
    feats = feature_stack(d0)

    # Fire mask AND observation mask for each day we care about.
    # Ilgilendigimiz her gun icin yangin maskesi VE gozlem maskesi.
    fire_t,  obs_t  = fire_and_obs(d0)
    fire_n1, obs_n1 = fire_and_obs(d0.advance(1, 'day'))
    fire_n2, obs_n2 = fire_and_obs(d0.advance(2, 'day'))

    fire_t  = fire_t.rename('fire')
    fire_n1 = fire_n1.rename('fire_next')
    fire_n2 = fire_n2.rename('fire_next2')

    # Temporal context. A single day's detection cannot distinguish a fire that has
    # burned continuously for three days from an isolated thermal anomaly, yet the
    # two behave completely differently on day t+1.
    # Zamansal baglam. Tek gunluk bir tespit, uc gundur yanan bir yangini tek
    # seferlik bir termal anomaliden ayirt edemez.
    fire_p1 = daily_fire_mask(d0.advance(-1, 'day')).rename('fire_prev1')
    fire_p2 = daily_fire_mask(d0.advance(-2, 'day')).rename('fire_prev2')

    # `valid` now means: every environmental input was observed AND MODIS actually
    # looked at this pixel today. v3 only checked the environmental band masks, so a
    # pixel hidden by cloud in the fire product still counted as valid.
    # `valid` artik: tum cevresel girdiler gozlendi VE MODIS bugun bu piksele
    # gercekten bakti. v3 yalnizca cevresel bant maskelerine bakiyordu.
    env_valid = feats.mask().reduce(ee.Reducer.min()).toFloat()
    valid = env_valid.multiply(obs_t).rename('valid')

    valid_n1 = obs_n1.rename('valid_next')
    valid_n2 = obs_n2.rename('valid_next2')

    stack = (feats.addBands([fire_p2, fire_p1, fire_t,
                             fire_n1, fire_n2,
                             valid, valid_n1, valid_n2])
                  .select(ALL_BANDS))

    # unmask(sameFootprint=False) makes the image defined everywhere, guaranteeing
    # that neighborhoodToArray always returns a full 65 x 65 array. Without it,
    # patches near a mask edge are truncated and fail to parse during training.
    # unmask(sameFootprint=False) goruntuyu her yerde tanimli yapar; boylece
    # neighborhoodToArray her zaman tam 65 x 65 dizi dondurur.
    return stack.unmask(0, False).reproject(PROJ)


## 5 — Sampling and export / Örnekleme ve dışa aktarım

Up to `MAX_POINTS_PER_DAY` burning pixels are drawn per fire day with
`stratifiedSample`, which prevents a single very large event from dominating the archive.

**Implementation note.** `stratifiedSample` attaches a scalar property named `fire` to
each sampled point. That property collides with the 65 × 65 `fire` *band* produced by
`neighborhoodToArray` and silently reduces it to a 1 × 1 scalar, which corrupts the
channel. All point properties are therefore discarded and only the geometry is retained;
longitude and latitude are re-attached under names that cannot collide with any band.

Her yangın gününde `stratifiedSample` ile en fazla `MAX_POINTS_PER_DAY` yanan piksel
seçilir; bu, tek bir çok büyük olayın arşive hâkim olmasını engeller.

**Uygulama notu.** `stratifiedSample`, örneklenen her noktaya `fire` adlı skaler bir
özellik ekler. Bu özellik, `neighborhoodToArray`'in ürettiği 65 × 65 `fire` *bandıyla*
çakışır ve onu sessizce 1 × 1 skalere indirger; kanal bozulur. Bu yüzden noktaların tüm
özellikleri atılır ve yalnızca geometri korunur; boylam ve enlem, hiçbir bantla
çakışmayacak adlarla yeniden eklenir.

In [ ]:
def fire_points(date, max_points):
    '''Sample burning pixels for one day, returning geometry-only features.'''
    fire_t = daily_fire_mask(date).selfMask().toInt()
    pts = fire_t.stratifiedSample(
        numPoints=max_points, classBand='fire',
        region=REGION.geometry(), scale=SCALE, projection=PROJ,
        seed=SEED, geometries=True, dropNulls=True)
    # Drop every property; keep geometry. Re-attach coordinates under safe names.
    # Tum ozellikleri at, geometriyi tut. Koordinatlari guvenli adlarla yeniden ekle.
    return pts.map(lambda f: ee.Feature(f.geometry()).set(
        'lon', f.geometry().coordinates().get(0),
        'lat', f.geometry().coordinates().get(1)))


def export_day(date_str):
    '''Submit one day's patches to Drive as a gzip TFRecord shard.'''
    date  = ee.Date(date_str)
    pts   = fire_points(date, MAX_POINTS_PER_DAY)
    stack = build_sample_image(date)

    arrays  = stack.neighborhoodToArray(ee.Kernel.square(PATCH_RADIUS, 'pixels'))
    samples = arrays.sampleRegions(collection=pts, scale=SCALE,
                                   projection=PROJ, geometries=False)
    samples = samples.map(lambda f: f.set('date', date_str))

    # File name stays schema-agnostic so the downloaded shard keeps the name the
    # loader expects; the TASK description carries the version so that the resume
    # scan in the next section is version-scoped.
    # Dosya adi ayni kalir (yukleyici bunu bekler); GOREV aciklamasi surum tasir,
    # boylece devam taramasi surume gore ayrisir.
    name = 'firespread_' + date_str.replace('-', '')
    desc = f'firespread_{VERSION_TAG}_' + date_str.replace('-', '')
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=desc,
        folder=DRIVE_FOLDER,
        fileNamePrefix=name,
        fileFormat='TFRecord',
        selectors=ALL_BANDS + META_COLS)
    task.start()
    return task

## 6 — Day selection / Gün seçimi

A day is exported only if Türkiye recorded at least `MIN_FIRE_PIXELS` active fire pixels.
Testing that one day at a time would cost roughly 1100 round trips to the Earth Engine
servers, so the counts are batched.

**How the batching must be done.** The obvious approach — mapping `reduceRegion` over a
list of dates — opens *one aggregation per day* and immediately trips Earth Engine's
`Too many concurrent aggregations` limit; the client then retries into the same wall and
the whole span fails. Instead each day is made a separate **band** of a single image, so
one `reduceRegion` call returns every count at once. That is one aggregation per chunk
rather than one per day.

If a chunk still fails, it is bisected and each half retried, down to single days, so a
single problematic date cannot discard a whole span.

Bir gün, yalnızca Türkiye'de en az `MIN_FIRE_PIXELS` aktif yangın pikseli kaydedildiyse
dışa aktarılır. Bunu günlük test etmek Earth Engine sunucularına yaklaşık 1100 gidiş-dönüş
demektir, bu yüzden sayımlar gruplanır.

**Gruplamanın nasıl yapılması gerektiği.** Akla ilk gelen yaklaşım — `reduceRegion`'ı bir
tarih listesi üzerinde map etmek — *gün başına bir toplulaştırma* açar ve anında Earth
Engine'in `Too many concurrent aggregations` sınırına takılır; istemci de aynı duvara
yeniden denemeler yapar ve tüm aralık başarısız olur. Bunun yerine her gün tek bir
görüntünün ayrı bir **bandı** yapılır; böylece tek bir `reduceRegion` çağrısı tüm sayımları
birden döndürür. Bu, gün başına değil grup başına bir toplulaştırmadır.

Bir grup yine de başarısız olursa ikiye bölünür ve her yarısı yeniden denenir; tek bir
sorunlu tarih tüm aralığı çöpe atamaz.

In [ ]:
import time

CHUNK_DAYS = 10   # days per aggregation / toplulastirma basina gun sayisi

# ─────────────────── DOCUMENTED SENSOR OUTAGES / BELGELENMIS SENSOR KESINTILERI ──
# Days on which Terra MODIS did not acquire data cannot produce a valid `lst` or
# `ndvi` channel: MOD11A1 has a 3-day compositing window, so a window falling entirely
# inside an outage yields an empty ImageCollection, and .mean() on it returns a
# band-less image that fails at .rename(). Such days are excluded rather than forced
# through, because filling them from stale observations would fabricate two of the
# fourteen input channels.
#
# Terra MODIS'in veri toplamadigi gunler gecerli bir `lst` veya `ndvi` kanali
# uretemez: MOD11A1 3 gunluk bir derleme penceresi kullanir; pencere tamamen kesinti
# icine duserse ImageCollection bos kalir, .mean() bantsiz bir goruntu dondurur ve
# .rename() hata verir. Bu gunler zorlanmak yerine dislanir, cunku onlari eski
# gozlemlerle doldurmak on dort girdi kanalindan ikisini uydurmak olurdu.
#
# Reference / Kaynak: LP DAAC, "Terra Constellation Exit & Data Outage,
# October 10-19, 2022". Acquisition ceased 10 Oct 2022 for retrograde manoeuvres on
# 12 and 19 Oct; instruments were still recovering through 21 Oct.
KNOWN_OUTAGES = [
    (datetime.date(2022, 10, 10), datetime.date(2022, 10, 22)),
]


def in_known_outage(day):
    return any(lo <= day <= hi for lo, hi in KNOWN_OUTAGES)


def _transient(exc):
    '''True if an Earth Engine error is a capacity limit worth retrying.'''
    m = str(exc).lower()
    return any(k in m for k in
               ('concurrent', 'too many', 'quota', 'timed out', 'try again', 'backend'))


def chunk_fire_counts(first_day, n_days):
    '''Daily active-fire pixel counts for a short span, in ONE server aggregation.

    Gunluk aktif yangin piksel sayilari, TEK sunucu toplulastirmasiyla.

    Each day becomes a separate BAND of one image, so a single reduceRegion returns
    every count together. Mapping reduceRegion over a list of dates instead opens one
    aggregation per day and trips the 'Too many concurrent aggregations' limit.
    Her gun tek bir goruntunun ayri bir BANDI olur; boylece tek reduceRegion tum
    sayimlari birlikte dondurur. reduceRegion'i bir tarih listesi uzerinde map etmek
    gun basina bir toplulastirma acar ve 'Too many concurrent aggregations' sinirina
    takilir.
    '''
    start = ee.Date(first_day.isoformat())
    stack = ee.Image.cat([daily_fire_mask(start.advance(i, 'day')).rename(f'd{i:02d}')
                          for i in range(n_days)])

    last = None
    for attempt in range(4):
        try:
            res = stack.reduceRegion(
                reducer=ee.Reducer.sum(), geometry=REGION.geometry(), scale=SCALE,
                maxPixels=1e10, bestEffort=False, tileScale=8).getInfo()
            return [(first_day + datetime.timedelta(days=i), res.get(f'd{i:02d}') or 0)
                    for i in range(n_days)]
        except Exception as exc:
            last = exc
            if not _transient(exc):
                raise
            wait = 8 * (2 ** attempt)
            print(f'      capacity limit, waiting {wait}s / kapasite siniri, {wait}s bekleniyor')
            time.sleep(wait)
    raise last


def counts_for_span(first_day, n_days, depth=0):
    '''Counts for a span, bisecting on failure so one bad date cannot lose the rest.'''
    try:
        return chunk_fire_counts(first_day, n_days)
    except Exception as exc:
        if n_days <= 1 or depth >= 5:
            print(f'      {first_day}: giving up / vazgecildi ({str(exc)[:60]})')
            return []
        half = n_days // 2
        print(f'      {first_day} +{n_days}d failed, bisecting / bolunuyor')
        return (counts_for_span(first_day, half, depth + 1)
                + counts_for_span(first_day + datetime.timedelta(days=half),
                                  n_days - half, depth + 1))


def season_bounds(year):
    '''First and last calendar day of the fire season, clipped to available data.'''
    first = datetime.date(year, FIRE_SEASON[0], 1)
    m = FIRE_SEASON[1]
    last = (datetime.date(year + 1, 1, 1) if m == 12
            else datetime.date(year, m + 1, 1)) - datetime.timedelta(days=1)
    return first, min(last, LAST_SAMPLE_DATE)


def candidate_days():
    '''All fire-season days with sufficient activity, within the available archive.'''
    out = []
    for year in YEARS:
        first, last = season_bounds(year)
        if first > last:
            print(f'  {year}: outside available archive / mevcut arsivin disinda')
            continue
        kept_year, scanned, excluded = 0, 0, 0
        day = first
        while day <= last:
            n = min(CHUNK_DAYS, (last - day).days + 1)
            counts = counts_for_span(day, n)
            scanned += len(counts)
            hits = [d for d, c in counts if c >= MIN_FIRE_PIXELS]
            usable = [d for d in hits if not in_known_outage(d)]
            excluded += len(hits) - len(usable)
            out.extend(usable)
            kept_year += len(usable)
            day += datetime.timedelta(days=n)
        note = f'  ({excluded} excluded, sensor outage / sensor kesintisi)' if excluded else ''
        print(f'  {year}: {kept_year:>3} fire days of {scanned} scanned '
              f'/ {scanned} gunde {kept_year} yangin gunu{note}')
    return sorted(d.isoformat() for d in out)


print('Scanning fire seasons / yangin sezonlari taraniyor ...\n')
DAYS = candidate_days()
print(f'\nTotal candidate fire days / toplam aday yangin gunu: {len(DAYS)}')
if DAYS:
    print(f'Range / aralik: {DAYS[0]} .. {DAYS[-1]}')
else:
    print('No days selected. Check the errors above before continuing.')
    print('Hic gun secilmedi. Devam etmeden once yukaridaki hatalari kontrol edin.')

## 7 — Resumable export / Yeniden başlatılabilir dışa aktarım

Generating the full archive takes longer than a single Colab session. Days whose shard is
already present in Drive, or whose export task is already queued or running, are skipped.
Re-running this notebook therefore continues from where the previous session stopped.

Tam arşivi üretmek tek bir Colab oturumundan uzun sürer. Parçası Drive'da hâlihazırda
bulunan veya dışa aktarım görevi zaten kuyruğa alınmış ya da çalışan günler atlanır. Bu
nedenle bu notebook'u yeniden çalıştırmak, önceki oturumun bıraktığı yerden devam eder.

In [ ]:
import os, re

def completed_days():
    '''Days already exported to Drive, or currently queued / running in Earth Engine.'''
    done = set()

    # (a) Shards already written to Drive. / Drive'a yazilmis parcalar.
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        folder = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
        if os.path.isdir(folder):
            for fn in os.listdir(folder):
                m = re.match(r'firespread_(\d{8})', fn)
                if m:
                    s = m.group(1)
                    done.add(f'{s[:4]}-{s[4:6]}-{s[6:]}')
            print(f'Drive: {len(done)} shard(s) already present / parca zaten mevcut')
        else:
            print(f'Drive: folder {DRIVE_FOLDER} not found yet / klasor henuz yok')
    except Exception as exc:
        print(f'Drive scan skipped / Drive taramasi atlandi: {exc}')

    # (b) Tasks already submitted this or a previous session, for THIS schema
    #     version only. A v2 task named 'firespread_20220717' must not mark
    #     2022-07-17 as done for v3.
    #     Bu veya onceki oturumda gonderilmis gorevler — SADECE bu surum icin.
    try:
        active = 0
        for op in ee.data.listOperations():
            meta = op.get('metadata', {})
            state = meta.get('state', '')
            desc  = meta.get('description', '')
            m = re.match(rf'firespread_{VERSION_TAG}_(\d{{8}})$', desc)
            if m and state in ('PENDING', 'RUNNING', 'SUCCEEDED'):
                s = m.group(1)
                done.add(f'{s[:4]}-{s[4:6]}-{s[6:]}')
                if state in ('PENDING', 'RUNNING'):
                    active += 1
        print(f'Earth Engine: {active} task(s) pending or running / bekleyen veya calisan')
    except Exception as exc:
        print(f'Task scan skipped / gorev taramasi atlandi: {exc}')

    return done


DONE      = completed_days()
REMAINING = [d for d in DAYS if d not in DONE]

print(f'\nCompleted / tamamlanan : {len(DAYS) - len(REMAINING)}')
print(f'Remaining / kalan      : {len(REMAINING)}')
print(f'This run / bu calistirma: {min(len(REMAINING), SUBMIT_LIMIT)} '
      f'(SUBMIT_LIMIT = {SUBMIT_LIMIT})')

In [ ]:
submitted = []
for day in REMAINING[:SUBMIT_LIMIT]:
    try:
        export_day(day)
        submitted.append(day)
        print(f'  {day}  submitted / gonderildi')
    except Exception as exc:
        print(f'  {day}  FAILED / BASARISIZ: {exc}')

print(f'\n{len(submitted)} export task(s) submitted to Drive/{DRIVE_FOLDER}.')
print(f'{len(REMAINING) - len(submitted)} day(s) still remaining / gun hala kaldi.')
if len(REMAINING) > len(submitted):
    print('Re-run cells 7 and 8 to continue. / Devam icin 7 ve 8. hucreleri tekrar calistirin.')
print('\nMonitor: https://code.earthengine.google.com/tasks')

## 8 — Progress monitoring / İlerleme takibi

Re-run this cell periodically. Earth Engine processes a limited number of tasks
concurrently, so a large batch will complete over several hours.

Bu hücreyi düzenli aralıklarla yeniden çalıştırın. Earth Engine aynı anda sınırlı sayıda
görev işler; büyük bir grup birkaç saat içinde tamamlanır.

In [ ]:
import re
from collections import Counter

states = Counter()
failures = []
for op in ee.data.listOperations():
    meta = op.get('metadata', {})
    if not re.match(r'firespread_\d{8}$', meta.get('description', '')):
        continue
    st = meta.get('state', 'UNKNOWN')
    states[st] += 1
    if st == 'FAILED':
        # The failure reason lives at the TOP level of the operation, not inside
        # `metadata`; reading meta['error'] silently yields an empty string.
        # Hata nedeni islemin UST duzeyindedir, `metadata` icinde degil;
        # meta['error'] okumak sessizce bos dize dondurur.
        err = (op.get('error') or {}).get('message') or '(no message returned)'
        failures.append((meta.get('description'), err))

print('Export task states / disa aktarim gorev durumlari')
print('-' * 50)
for st, n in sorted(states.items()):
    print(f'  {st:<12} {n:>5}')

if failures:
    print(f'\n{len(failures)} failed task(s) / basarisiz gorev — first 10:')
    for desc, msg in failures[:10]:
        print(f'  {desc}: {msg[:110]}')
    print('\nFailed days are not recorded as complete and will be retried '
          'on the next run of cell 8.')
    print('Basarisiz gunler tamamlanmis sayilmaz; 8. hucrenin bir sonraki '
          'calistirilmasinda yeniden denenir.')

## 9 — Retrieving the archive / Arşivi indirme

> **Keep each archive version in its own directory.** A v3 shard has 19 input bands;
> a v2 shard has 14. Mixing them in one directory makes the channel axis inconsistent
> and the loader will reject the short records.
>
> **Her arşiv sürümünü kendi dizininde tutun.** Bir v3 parçası 19 girdi bandı taşır,
> bir v2 parçası 14. Aynı dizinde karıştırmak kanal eksenini tutarsız hâle getirir ve
> yükleyici kısa kayıtları reddeder.

When every task has reached `SUCCEEDED`, download the contents of
Drive → `GEE_FireSpread_v4/` into `data/spread_v4/`:

Tüm görevler `SUCCEEDED` durumuna ulaştığında, Drive → `GEE_FireSpread_v4/` klasörünün
içeriğini `data/spread_v4/` dizinine indirin:

```bash
mkdir -p data/spread_v4     # v3 shards live here / v3 parçaları buraya
# data/spread_v4/    stays as the v2 archive / v2 arşivi olarak kalır
```

Then convert to a memory-mapped local cache:

Ardından belleğe eşlenmiş yerel önbelleğe dönüştürün:

```bash
python src/tfrecord_to_npy.py --verify   # integrity report / bütünlük raporu
python src/train.py                      # train the U-Net / U-Net'i eğit
```

The conversion step is not optional. It is what allows the training loop to read patches
without decompressing gzip on every epoch, and it produces the per-channel normalisation
statistics from the training split alone.

Dönüştürme adımı isteğe bağlı değildir. Eğitim döngüsünün her epoch'ta gzip açmadan yama
okumasını sağlayan ve kanal başına normalizasyon istatistiklerini yalnızca eğitim
bölmesinden üreten adım budur.

---

> ### Nereye indirilecek — DİKKAT
>
> Bu arşiv **`data/spread_v4/`** klasörüne gider.
>
> **`data/spread/` içine KOYMAYIN** — orada v2 arşivi (17 bant) duruyor. Bant
> sırası sözleşmedir ve yükleyici kanal eksenini yalnızca bu sıradan kurar; farklı
> şemalar aynı klasördeyse kayıtlar reddedilmez, **sessizce yanlış yorumlanır** ve
> eğitim bozulur.
>
> Yükleyici artık her parçanın şemasını gerçek alan adlarından tespit edip karışık
> bir klasörü hata vererek durduruyor — ama yine de doğru klasöre koymak en iyisi.


## 10 — Visual inspection / Görsel inceleme

A sanity check on a single day: today's detections in red, tomorrow's in yellow. The
spatial offset between them is the signal the model is asked to learn.

Tek bir gün üzerinde akıl sağlığı kontrolü: bugünün tespitleri kırmızı, yarınınkiler
sarı. Aralarındaki uzamsal kayma, modelden öğrenmesi istenen sinyaldir.

In [ ]:
# 2021-07-29: second day of the Manavgat fire. / Manavgat yanginlarinin ikinci gunu.
DEMO_DATE = '2021-07-29'

fire_t    = daily_fire_mask(DEMO_DATE)
fire_next = daily_fire_mask(ee.Date(DEMO_DATE).advance(1, 'day'))
feats     = feature_stack(DEMO_DATE)
valid     = feats.mask().reduce(ee.Reducer.min())

m = geemap.Map()
m.centerObject(REGION, 6)
m.addLayer(feats.select('ndvi'),
           {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}, 'NDVI')
m.addLayer(feats.select('wind_speed'),
           {'min': 0, 'max': 12, 'palette': ['white', 'blue', 'purple']}, 'Wind speed')
m.addLayer(valid, {'min': 0, 'max': 1, 'palette': ['black', 'white']},
           'Valid observations', False)
m.addLayer(fire_t.selfMask(),    {'palette': ['red']},    'Fire, day t')
m.addLayer(fire_next.selfMask(), {'palette': ['yellow']}, 'Fire, day t+1')
m